# Individual Assignment I
## Data Preparation and Pipeline Design
### Viktoriia Stepanenko

The tasks in this notebook are executed in a deliberate order that reflects 
three core principles: respecting the temporal logic of prediction (no 
information available after prediction time may be used as input), 
preserving the independence of training, validation, and test sets, and 
ensuring that all data-driven transformations are fitted exclusively on 
training data. This ordering prevents both preprocessing leakage and target 
leakage, and guarantees that model evaluation reflects realistic performance.

## Task 1 – Identify the Prediction Target

**Task requirements:**
- Inspect the dataset and identify the correct target variable.
- Justify why it is the appropriate prediction objective.
- Identify at least two variables that could superficially appear to be 
  valid targets and explain why they should not be used.

The central concept that guides target identification is the **prediction 
time constraint**: in a supervised learning setting, the target variable 
must represent an outcome that is *unknown at the moment a prediction is 
requested*, while all input features must be *observable before that 
moment*. Violating this constraint, whether in the target or in the 
features, introduces **target leakage** and produces a model that cannot 
generalise to real-world deployment.

In this dataset, the correct target variable is `y`, a binary column 
encoding whether a given client subscribed to a term deposit following 
a telemarketing contact (`yes` / `no`). This is the appropriate prediction 
objective for two reasons. First, it directly represents the business 
outcome the bank is trying to anticipate: before deciding whether to 
contact a client, the bank wants to estimate the probability of a 
successful subscription. Second, `y` is observed strictly *after* the 
interaction concludes — it is never available at prediction time — which 
satisfies the formal requirement for a valid label in the supervised 
learning framework.

Two other variables could appear to be relevant targets but must not be 
treated as such:

**`duration`** records the length of the last phone call in seconds. It 
may superficially seem like a useful proxy for client engagement — longer 
calls plausibly correlate with higher interest. However, call duration is 
only measurable *after the call has ended*, meaning it is unavailable at 
the moment the bank decides whether to place a call. More critically, 
duration is strongly correlated with the outcome itself: calls of zero 
seconds always result in `y = no`, and longer calls are systematically 
associated with subscriptions. Treating it as an input feature — let alone 
a target — would constitute target leakage and render the model useless in 
production. `duration` will be dropped entirely before any further 
processing.

**`pdays`** records the number of days since the client was last contacted 
in a *previous* campaign, with the value `999` used as a sentinel to 
indicate that the client was never previously contacted. While `pdays` 
remains as an input feature (it is observable before the call), it could 
be mistakenly treated as a target because it summarises a key aspect of 
the client relationship. It is not, however, the outcome the bank is 
trying to predict — it describes history, not the future subscription 
decision — and so it does not qualify as a valid prediction target.

In [1]:
# Task 1 – Identify Prediction Target

TARGET = 'y'

# 'duration' is post-call information only available after the outcome
# is already determined. It must be removed before any other step.
LEAKY_FEATURES = ['duration']

print(f"Target variable : '{TARGET}'")
print(f"Leaky features  : {LEAKY_FEATURES}")
print(f"\n'duration' will be dropped at load time to prevent any risk")
print(f"of it influencing downstream transformations.")

Target variable : 'y'
Leaky features  : ['duration']

'duration' will be dropped at load time to prevent any risk
of it influencing downstream transformations.


## Task 2 – Data Loading and Exploration

**Task requirements:**
- Load the dataset and inspect its structure (shape, types, summary statistics).
- Identify numerical and categorical variables.
- Analyze the target variable distribution and comment on class imbalance.
- Detect explicit and implicit missing values.
- Visualize at least two numerical and two categorical variables, each supporting a specific observation.
- Identify at least one variable requiring special consideration before modelling.

Exploratory Data Analysis (EDA) serves a precise purpose: to surface structural 
properties of the data that will directly inform every subsequent pipeline decision. 
Each inspection below is conducted with a specific question in mind, not as a 
mechanical checklist.

**Important technical note:** `bank-additional.csv` uses semicolons as field 
separators, not commas. This is common in European datasets where commas serve 
as decimal separators. Failing to specify `sep=';'` would load the entire row 
as a single column.